# Task 1: OAuth & U2M Authentication

# Databricks CLI: Core Concepts & Windows Reference Guide

---

## Part 1: Core Concepts & Assignment Overview

### What is Databricks CLI?
The Databricks Command Line Interface (CLI) allows you to automate and manage your Databricks workspace from a Windows terminal (PowerShell/CMD) without using the web UI.

### Key Difference: U2M vs M2M Authentication

| Feature | U2M (User-to-Machine) | M2M (Machine-to-Machine) |
| :--- | :--- | :--- |
| **Who uses it?** | Human Developers / Students | Automated Scripts / CI/CD Pipelines |
| **Login Method** | Web browser login (databricks auth login) | Client ID + Client Secret |
| **Identity** | Your personal user account | Databricks Service Principal |
| **Use Case** | Local testing, daily tasks, assignments | Scheduled jobs, GitHub Actions, Azure DevOps |

---

## Part 2: Task 1 - Basic Setup & U2M Login

### Step 1: Install CLI on Windows
Open **PowerShell** as Administrator and run:
- `winget install Databricks.DatabricksCLI`

### Step 2: Authenticate using U2M
Run this command in PowerShell to log in via browser:
- `databricks auth login --host https://<your-workspace-url> --profile DEFAULT
`
### Step 3: Verify Authentication
- `databricks auth profiles`
- `databricks fs ls dbfs:/`

---

# Task 2: Bundle Initialize

## Task 2: Initialize a Declarative Automation Bundle Project

### Objective
Create a Databricks Asset Bundle (DAB) project containing one job resource, using the CLI's scaffolding command.

### Command Used
```bash
databricks bundle init
```

### Steps Performed
1. Ran `databricks bundle init` from the terminal.
2. Selected the default Python template when prompted by the interactive wizard.
3. Provided a project name: `my_first_bundle`.
4. The CLI generated the following folder structure automatically:

```
my_first_bundle/
├── databricks.yml
├── src/
│   └── notebook.ipynb
├── resources/
│   └── my_first_bundle.job.yml
└── README.md
```

### Bundle Configuration (`databricks.yml`)
```yaml
bundle:
  name: my_first_bundle

include:
  - resources/*.yml

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: https://dbc-0cc14214-f5e5.cloud.databricks.com/browse/folders/workspace?o=7474655513463998
```

### Job Resource (`resources/my_first_bundle.job.yml`)
```yaml
resources:
  jobs:
    my_job:
      name: first_job_dab
      tasks:
        - task_key: main_task
          notebook_task:
            notebook_path: ../src/notebook.ipynb
```

### Outcome
- A working bundle project was scaffolded with one job resource (`my_job`).
- The project structure separates the bundle config (`databricks.yml`) from resource definitions (`resources/`), which keeps things organized as more jobs/pipelines get added later.
- `dev` was set as the default target, meaning any `databricks bundle deploy` command without an explicit `-t` flag will deploy here.

---

# Task 3: Validate & Deploy

## Task 3: Validate and Deploy the Bundle

### Objective
Confirm the bundle configuration is syntactically correct, deploy it to the `dev` target, and verify the job was created in the workspace.

### Step 1 — Validate
```bash
databricks bundle validate
```

**What this does:**
- Parses `databricks.yml` and all included resource files.
- Checks for schema errors (wrong field names, bad indentation, missing required keys).
- Resolves variable references and confirms the target workspace is reachable.

**Sample output:**
```
Name: first_job_dab
Target: dev
Workspace:
  Host: https://dbc-0cc14214-f5e5.cloud.databricks.com/browse/folders/workspace?o=7474655513463998
  User: bhaveshyadav@cyntexa.com

Validation OK!
```

### Step 2 — Deploy to Dev
```bash
databricks bundle deploy -t dev
```

**What this does:**
- Uploads bundle source files (notebooks, code) to a workspace directory under `/Workspace/Users/bhaveshyadav/.bundle/my_first_bundle/dev/`.
- Creates/updates the job resource defined in `resources/first_job_dab.job.yml` inside the target workspace.
- Since `mode: development` is set, the deployed job is prefixed with `[dev bhaveshyadav]` and treated as a personal, disposable deployment (safe for iteration).

**Sample output:**
```
Uploading bundle files to /Workspace/Users/bhaveshyadav@cyntexa.com/.bundle/first_job_dab/dev/files...
Deploying resources...
Updating deployment state...
Deployment complete!
```

### Step 3 — Confirm in Workspace UI
1. Logged into the Databricks workspace.
2. Navigated to **Workflows → Jobs**.
3. Located the deployed job: `[dev bhaveshyadav] first_job_dab`.
4. Opened the job to confirm the task (`main_task`) correctly points to the uploaded notebook.

### Outcome
- `databricks bundle validate` passed with no errors, confirming the YAML structure and workspace connectivity were correct.
- `databricks bundle deploy -t dev` successfully created the job in the workspace.
- The job was visually confirmed in the **Workflows** tab, verified by name and task configuration.

---

## Summary

| Task | Command(s) | Result |
|------|-----------|--------|
| Task 2 | `databricks bundle init` | Bundle project scaffolded with one job resource |
| Task 3 | `databricks bundle validate` → `databricks bundle deploy -t dev` | Config validated, job deployed and confirmed in workspace |

# Intermediate Tasks 

## Task 4: Add a Second Target (Staging) to `databricks.yml`

### Objective
Extend the bundle configuration with a second deployment target (`staging`), using a different workspace host and a `run_as` identity, then deploy to it.

### Constraint Encountered: Databricks Free Edition Limitation
While attempting this task, I found that **Databricks Free Edition allows only one workspace and one metastore per account**, with no access to the account console or account-level APIs. This means:

- A second, physically separate workspace URL cannot be provisioned under a Free Edition account.
- Service principals cannot be created either, since their setup requires account console access (relevant for `run_as` in a real production setup).

### Design Decision
Since a true second workspace isn't available, I simulated the `staging` target within the same Free Edition workspace by:
- Pointing both `dev` and `staging` at the same `workspace.host`.
- Differentiating them using `mode` (`development` vs `production`) and a distinct job-name prefix, so deployments remain logically separated and don't collide in the workspace UI.
- Using `run_as.user_name` (my own account) instead of a service principal, since service principals aren't available in Free Edition.

In a real multi-workspace setup (paid tier), `staging.workspace.host` would point to an entirely separate workspace URL (e.g. a dedicated staging environment), and `run_as` would reference a service principal rather than a personal user account.

### Updated `databricks.yml`
```yaml
bundle:
  name: my_first_bundle

include:
  - resources/*.yml

targets:
  dev:
    mode: development
    default: true
    workspace:
      host: https://<your-free-edition-workspace-url>

  staging:
    mode: production
    workspace:
      host: https://<your-free-edition-workspace-url>   # same host — Free Edition limitation, see note above
    run_as:
      user_name: bhaveshyadav@cyntexa.com   # service principal not available in Free Edition
    resources:
      jobs:
        my_job:
          name: "[staging] my-first-job"
```

### Deployment Command
```bash
databricks bundle deploy -t staging
```

**What this does:**
- Deploys the same job resource under the `staging` target's naming/mode settings.
- Because `mode: production` is set (unlike `dev`'s `development` mode), Databricks enforces production-safety behaviors — e.g., the job is not automatically tied to my personal user identity for pausing/editing the way dev jobs are, and locking behavior is stricter.

### Verification
1. Logged into the Databricks workspace.
2. Navigated to **Workflows → Jobs**.
3. Confirmed two separate job entries:
   - `[dev bhaveshyadav] my-first-job` (from Task 3)
   - `[staging] my-first-job` (from this task)
4. Both jobs point to the same underlying workspace but are clearly distinguishable by name and were deployed independently via their respective targets.

### Outcome
- Successfully demonstrated the *concept* of multi-target deployment (dev vs. staging) using `databricks.yml`, despite the Free Edition's single-workspace constraint.
- Documented the limitation transparently and explained how the setup would differ in a full production environment with multiple real workspaces and service principals.

---

## Summary

| Item | Real Production Setup | Free Edition Simulation (this submission) |
|------|------------------------|---------------------------------------------|
| Workspace host | Separate URL per target | Same URL, differentiated by `mode` + naming |
| `run_as` identity | Service principal | Personal user account |
| Job separation | Physical (different workspaces) | Logical (job name prefix + mode) |

# Task 5: M2M (Service Principal) Authentication

In [0]:
# client_id = a4fa772c-a894-4a06-a8f4-2fe54ccf4e25
# client_secret = dose936b84e50159a414702e2222996e4be9

## Task 5: Configure M2M Authentication and Use It for a Deploy Command

### Objective
Set up service principal (Machine-to-Machine) authentication for the Databricks CLI and use it — instead of personal OAuth U2M login — to run a bundle deploy command.

### Step 1 — Create a Service Principal
A service principal named `github-actions-sp` was created via the Databricks Account Console (Account Console → User Management → Service Principals → Add service principal), and added to the target workspace with the necessary roles.

### Step 2 — Generate Client ID and Client Secret
From the service principal's **Secrets** tab, an OAuth secret was generated:
- **Client ID:** `a4fa772c-a894-4a06-a8f4-2fe54ccf4e25`
- **Client Secret:** generated and stored securely (only shown once at creation time).

### Step 3 — Set Environment Variables (PowerShell)
Since the CLI was being run on Windows, `export` (bash syntax) does not apply. The correct PowerShell syntax used was:

```powershell
$env:DATABRICKS_HOST="https://dbc-0cc14214-f5e5.cloud.databricks.com"
$env:DATABRICKS_CLIENT_ID="a4fa772c-a894-4a06-a8f4-2fe54ccf4e25"
$env:DATABRICKS_CLIENT_SECRET="<client-secret>"
```

These variables are session-scoped — they only apply to the current PowerShell window and must be re-set if the terminal is closed and reopened.

### Step 4 — Run Deploy Using M2M Auth
```powershell
databricks bundle deploy -t dev
```
With `DATABRICKS_CLIENT_ID` / `DATABRICKS_CLIENT_SECRET` set, the CLI authenticates non-interactively as the service principal instead of opening a browser for U2M login — this is the exact mechanism a CI/CD pipeline (e.g., GitHub Actions) relies on, since no human is present to complete an interactive login.

## Outcome

- M2M authentication was successfully configured and used to trigger `databricks bundle deploy` non-interactively via `DATABRICKS_CLIENT_ID` / `DATABRICKS_CLIENT_SECRET`.
- The exercise surfaced four distinct permission boundaries that only appear when moving from single-user (U2M) development to a service-principal-driven (M2M) deployment flow: workspace folder ACLs, resource ownership rules, permission-level granularity (`CAN_RUN` vs `CAN_MANAGE`), and Unity Catalog data-access grants.
- Each of these maps directly to a real consideration when designing a CI/CD pipeline for Databricks: resources should be created by the automation identity from the start, granted the correct permission level for both planning and execution, and given explicit Unity Catalog access separate from workspace-level access.

---

## Summary

| Issue | Layer | Fix |
|---|---|---|
| Folder access denied | Workspace ACL (personal folder) | Moved `root_path` to `/Shared/...` |
| Owner-change blocked | Job/Pipeline ownership | Removed `run_as`; let SP create resources fresh |
| Manage vs Run confusion | Bundle permission level | Used `CAN_MANAGE`, not `CAN_RUN` |
| Catalog access denied | Unity Catalog (data layer) | `GRANT USE CATALOG` / `USE SCHEMA` to SP |

# Task 6: GitHub Actions — Validate on PR

## Task 6: Write a GitHub Actions Workflow That Validates on Every PR

### Objective
Create a GitHub Actions workflow that runs `databricks bundle validate` automatically on every pull request, without deploying anything.

### Repository Structure
```
Data_engineering_assignments/        (Git repo root)
├── .github/
│   └── workflows/
│       └── validate.yml
└── first_job_dab/                   (bundle project)
    ├── databricks.yml
    ├── resources/
    └── src/
```

The `.github/workflows/` folder must sit at the repository root for GitHub to detect it. Since the bundle lives in the `first_job_dab` subfolder, the workflow uses `working-directory` to run commands from the correct path.

### Workflow File (`.github/workflows/validate.yml`)
```yaml
name: Validate Bundle

on:
  pull_request:
    branches: [main]

jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Validate bundle
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
          DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
        working-directory: ./first_job_dab
        run: databricks bundle validate -t dev
```

### GitHub Repository Secrets
Service principal credentials are stored as encrypted repository secrets (Settings → Secrets and variables → Actions) rather than hardcoded in the workflow file:

| Secret Name | Purpose |
|---|---|
| `DATABRICKS_HOST` | Workspace URL |
| `DATABRICKS_CLIENT_ID` | Service principal client ID |
| `DATABRICKS_CLIENT_SECRET` | Service principal client secret |

This allows the workflow to authenticate non-interactively as the `github-actions-sp` service principal (M2M auth), the same mechanism configured in Task 5.

### Testing the Workflow
1. Created a new branch: `test-validate-workflow`.
2. Made a small change to `databricks.yml` and pushed the branch.
3. Opened a pull request from `test-validate-workflow` into `main`.
4. GitHub automatically triggered the "Validate Bundle" check on the PR, visible under the PR's **Checks** section and in the repo's **Actions** tab.

### Outcome
- The workflow runs `databricks bundle validate` on every pull request targeting `main`.
- No `deploy` step is included, so no resources are created or modified in the workspace — the workflow is purely a validation gate.
- A passing check (green ✅) confirms the bundle configuration is syntactically correct and the workspace is reachable before any code is merged, catching configuration errors early in the review process.

## Task 7: Extend GitHub Actions to Deploy to Prod Using OIDC

### Objective
Extend the CI/CD pipeline so that merging to `main` automatically deploys the bundle to `prod`, authenticating via OIDC instead of a stored client secret.

### Why OIDC Instead of a Stored Secret
In Task 5/6, the workflow authenticated using `DATABRICKS_CLIENT_ID` + `DATABRICKS_CLIENT_SECRET`, both stored as GitHub repository secrets. A stored secret is long-lived — it doesn't expire on its own and carries risk if leaked (e.g., through a misconfigured log or a compromised repo).

With OIDC, GitHub issues a short-lived identity token for each individual workflow run, scoped to that specific repository and branch. Databricks verifies this token against a pre-configured trust relationship (a federation policy) instead of checking a static secret. No `client_secret` needs to exist in GitHub at all.

### Step 1 — Federation Policy (Databricks Account Console)
A federation policy was configured on the `github-actions-sp` service principal to trust tokens issued by GitHub's OIDC provider, scoped to this repository's `main` branch:

```
Issuer:  https://token.actions.githubusercontent.com
Subject: repo:bhaveshyadav-DEV/Data_engineering_assignments:ref:refs/heads/main
```

Scoping the `Subject` to this exact repository and branch ensures only workflow runs from `main` in this specific repo can authenticate as the service principal — a workflow run from a fork or a different branch would be rejected.

### Step 2 — Workflow File (`.github/workflows/deploy-prod.yml`)
```yaml
name: Deploy to Prod

on:
  push:
    branches: [main]

permissions:
  id-token: write
  contents: read

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout repository
        uses: actions/checkout@v4

      - name: Install Databricks CLI
        uses: databricks/setup-cli@main

      - name: Deploy bundle to prod
        working-directory: ./first_job_dab
        env:
          DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
          DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}
        run: databricks bundle deploy -t prod
```

**Key points:**
- `permissions: id-token: write` is what allows GitHub to mint an OIDC token for this run — without it, no token is issued and authentication fails.
- Only `DATABRICKS_HOST` and `DATABRICKS_CLIENT_ID` are needed as secrets; there is no `DATABRICKS_CLIENT_SECRET` anywhere in this workflow.
- `working-directory: ./first_job_dab` points the CLI at the bundle folder, since the repository root also contains the `.github/` workflow files.

### Step 3 — Trigger and Verify
1. The Task 6 test pull request was merged into `main`.
2. The merge triggered a `push` event on `main`, which started the "Deploy to Prod" workflow automatically.
3. The run was verified under the repository's **Actions** tab, confirming the deploy step authenticated and completed without any client secret present in the workflow.

### Outcome
- Every merge to `main` now automatically deploys the bundle to `prod`.
- Authentication uses a short-lived, repo-and-branch-scoped OIDC token instead of a persistent stored secret, removing the risk and maintenance overhead (rotation, leakage) associated with long-lived credentials.
- Combined with Task 6's validate-on-PR workflow, the full pipeline is: PR opened → validate runs → PR reviewed and merged → prod deploy runs automatically via OIDC.

## Task 8: Design a Rollback Plan for a Broken Prod Deployment

### Objective
If a bundle deploy to `prod` breaks a job, define which CLI commands would be run to redeploy the previous working version quickly.

### Step 1 — Confirm What Broke
Check the currently deployed state and recent run history before taking any action:
```bash
databricks bundle summary -t prod
databricks jobs list-runs --job-id <job-id>
```
This confirms which deployment is live and what actually failed, avoiding a rollback based on assumption.

### Step 2 — Identify the Last Known-Good Commit
```bash
git log --oneline
```
Since every `prod` deployment corresponds to a specific Git commit, the previous working state is identified by finding the commit before the one that introduced the breaking change.

### Step 3 — Revert the Change
```bash
git revert <bad-commit-sha>
```
`git revert` creates a new commit that undoes the breaking change, rather than deleting or rewriting history. This keeps a clear, auditable record of what happened and why, which matters when a team needs to understand later what broke and how it was fixed.

### Step 4 — Redeploy
If deployment is CI/CD-driven (as configured in Task 7), pushing the revert commit to `main` is sufficient — the pipeline redeploys automatically:
```bash
git push origin main
```
If deploying manually:
```bash
databricks bundle deploy -t prod
```

### Step 5 — Verify the Fix
```bash
databricks bundle summary -t prod
databricks jobs run-now --job-id <job-id>
```
Confirms the reverted version is live and the job runs successfully again.

### Emergency Alternative — Manual Checkout
In a genuinely time-critical outage, a faster (but riskier) path is checking out the previous commit directly and deploying without waiting for a revert-and-merge cycle:
```bash
git checkout <previous-working-commit-sha>
databricks bundle deploy -t prod
git checkout main
```
This restores service immediately but leaves no clean commit-level record of the rollback, and risks the branch state and workspace state falling out of sync if not followed up carefully.

---

## Design Decision: Revert-and-Redeploy vs. Manual Direct Deploy

| Approach | Speed | Audit Trail | Risk |
|---|---|---|---|
| `git revert` + CI/CD push | Slower (goes through the pipeline/merge cycle) | Clean — the rollback itself is a visible, reviewable commit | Low |
| Manual `git checkout` + direct deploy | Fastest | None — no commit records the rollback happened | Higher — easy for branch and deployed state to drift apart |

**Recommendation:** Use `git revert` followed by the normal CI/CD deploy path as the default rollback method, since it preserves history and keeps the deployed state traceable to a specific commit at all times. Reserve the manual direct-deploy path for genuine production outages where minutes matter, and treat it as an interim fix — the corresponding revert commit should still be pushed afterward so the Git history reflects what is actually running in `prod`.

# Day 9 Assignment — Advanced Task Documentation

**Prepared by:** Bhavesh
**Task covered:** Task 9 (One-Page Onboarding Guide)

---

## Onboarding Guide: From a Local Edit to a Safe Production Deploy

*Welcome to the team! This guide walks through how a change to our Databricks pipeline moves from your laptop to production, and why each step exists.*

### 1. Local Development
You start by editing `databricks.yml` or a resource file (job/pipeline definition) on your machine. Before touching the workspace, check your work locally:
```bash
databricks bundle validate
```
This catches YAML/schema errors immediately, without needing to deploy anything.

### 2. Deploy to Your Personal Dev Environment
Once validation passes, deploy to the `dev` target to see your change running in an actual workspace:
```bash
databricks bundle deploy -t dev
```
This uses your personal login (OAuth U2M — the browser-based login you set up when you first installed the CLI). Because the target is set to `mode: development`, your job is automatically prefixed with your username, so it never collides with anyone else's dev deployment or with production.

### 3. Open a Pull Request
When you're happy with your change, push your branch and open a PR into `main`. This is where the safety net kicks in automatically — you don't need to remember any extra steps here.

### 4. Automated Validation (CI)
Opening the PR triggers a GitHub Actions workflow that runs `databricks bundle validate` against our workspace, using a service principal (`github-actions-sp`) rather than a personal login — the same validation you ran locally, but enforced automatically before anyone reviews your code. If this check fails, fix it and push again; the workflow reruns on every new commit to the PR.

### 5. Code Review and Merge
A teammate reviews your PR. Once approved and the validation check is green, it gets merged into `main`.

### 6. Automated Production Deploy (CD)
Merging to `main` triggers a second GitHub Actions workflow that deploys straight to `prod`:
```bash
databricks bundle deploy -t prod
```
This step authenticates using OIDC — GitHub issues a short-lived, repo-scoped token instead of relying on a stored secret, so there's no long-lived credential sitting in our repository that could leak or need periodic rotation.

### 7. Monitoring
After deployment, keep an eye on the job in the Databricks Workflows UI (or our alerting, if configured) to confirm it's running as expected on real data.

### 8. If Something Breaks: Rollback
If a prod deploy causes a job to fail, don't panic and don't hand-edit things in the UI. Find the last good commit and revert it:
```bash
git revert <bad-commit-sha>
git push origin main
```
Pushing the revert re-triggers the same CD pipeline from Step 6, redeploying the previous working version automatically. This keeps the rollback itself visible in our Git history, so anyone can see later what broke and how it was fixed.

---

### The Full Flow at a Glance

```
Local edit → validate → deploy to dev (U2M) → open PR
    → CI validates automatically (service principal)
    → review & merge
    → CD deploys to prod automatically (OIDC, no stored secret)
    → monitor
    → if broken: git revert → push → CD redeploys the fix
```

The core idea: **the CLI is for local iteration, the bundle lifecycle (`validate` → `deploy`) is the mechanism, and CI/CD is what makes that mechanism run consistently and safely without anyone needing to remember to do it by hand.**